In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.5lake_lab.silver_trips_pipeline"
)

df_ml = df.select(
    "trip_distance",
    "trip_duration_min",
    "tip_pct",
    F.hour("tpep_pickup_datetime").alias("pickup_hour"),
    F.dayofweek("tpep_pickup_datetime").alias("day_of_week")
).dropna()

print("Nombre de lignes :", df_ml.count())

display(df_ml.limit(10))


Nombre de lignes : 21847


trip_distance,trip_duration_min,tip_pct,pickup_hour,day_of_week
2.89,13.366667,24.14,20,1
0.6,3.016667,23.87,22,2
0.92,8.133333,12.14,20,6
2.14,12.900000,11.61,23,4
3.02,15.333333,2.58,21,3
1.17,5.100000,8.27,18,4
1.9,8.833333,27.31,21,3
1.8,18.550000,2.68,17,5
20.22,49.583333,15.74,19,6
1.2,9.100000,22.34,8,6


In [0]:

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Variables explicatives
features = [
    "trip_distance",
    "trip_duration_min",
    "pickup_hour",
    "day_of_week"
]

# Transformer les colonnes en vecteur
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features",
    handleInvalid="skip"
)

# Modèle Random Forest
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="tip_pct",
    numTrees=50,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])

# Séparer les données : 80 % entraînement, 20 % test
train_df, test_df = df_ml.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Entraîner
model = pipeline.fit(train_df)

# Prédire
predictions = model.transform(test_df)

display(predictions.select("tip_pct", "prediction").limit(10))


tip_pct,prediction
20.57,14.942610836772383
13.81,15.604301952964311
3.23,15.343416137340153
8.69,15.344825000849285
25.04,14.769004582816867
1.07,15.027158146467432
11.04,15.638460220175311
28.45,16.080834469501315
20.98,16.434073560383805
19.01,14.62506934523201


In [0]:

# Calcul du RMSE
rmse_evaluator = RegressionEvaluator(
    labelCol="tip_pct",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = rmse_evaluator.evaluate(predictions)

# Calcul du R²
r2_evaluator = RegressionEvaluator(
    labelCol="tip_pct",
    predictionCol="prediction",
    metricName="r2"
)

r2 = r2_evaluator.evaluate(predictions)

print("RMSE :", round(rmse, 4))
print("R² :", round(r2, 4))


RMSE : 8.6396
R² : -0.0087


In [0]:

import mlflow

with mlflow.start_run(run_name="RandomForest_Tip_Prediction") as run:

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 8)
    mlflow.log_param("train_ratio", 0.8)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    print("MLflow Run ID :", run.info.run_id)
    print("RMSE :", rmse)
    print("R² :", r2)


MLflow Run ID : 8cfe9d53c6c6412d8393df7e6b61a7fb
RMSE : 8.639550296887487
R² : -0.008706606508229076


In [0]:


import mlflow
import mlflow.spark
from mlflow.models import infer_signature

run_id = "8cfe9d53c6c6412d8393df7e6b61a7fb"

# Volume Unity Catalog existant
tmp_path = "/Volumes/workspace/5lake_lab/raw/mlflow_tmp"

# Signature du modèle pour Unity Catalog
signature = infer_signature(
    train_df.select(
        "trip_distance",
        "trip_duration_min",
        "pickup_hour",
        "day_of_week"
    ),
    predictions.select("prediction")
)

# Reprendre le run MLflow
with mlflow.start_run(run_id=run_id):

    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="random_forest_tip_model",
        dfs_tmpdir=tmp_path,
        signature=signature
    )

print("Modèle enregistré dans MLflow !")


2026/10/23 07:26:31 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/10/23 07:26:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-71c977d8-54f9-434c-9dfa-fc/tmpsbofvuwz/model, flavor: spark). Fall back to return ['pyspark==4.3.0']. Set logging level to DEBUG to see the full traceback. 


Modèle enregistré dans MLflow !


In [0]:

from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Définition explicite des colonnes d'entrée
input_schema = Schema([
    ColSpec("double", "trip_distance"),
    ColSpec("double", "trip_duration_min"),
    ColSpec("integer", "pickup_hour"),
    ColSpec("integer", "day_of_week")
])

# Définition de la sortie du modèle
output_schema = Schema([
    ColSpec("double", "prediction")
])

signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema
)

print(signature)


inputs: 
  ['trip_distance': double (required), 'trip_duration_min': double (required), 'pickup_hour': integer (required), 'day_of_week': integer (required)]
outputs: 
  ['prediction': double (required)]
params: 
  None



In [0]:

import mlflow
import mlflow.spark

run_id = "8cfe9d53c6c6412d8393df7e6b61a7fb"

with mlflow.start_run(run_id=run_id):

    model_info = mlflow.spark.log_model(
        spark_model=model,
        artifact_path="random_forest_tip_model_v2",
        dfs_tmpdir="/Volumes/workspace/5lake_lab/raw/mlflow_tmp",
        signature=signature
    )

print("Modèle enregistré avec signature corrigée !")
print("Model URI :", model_info.model_uri)


2026/10/23 07:30:46 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-71c977d8-54f9-434c-9dfa-fc/tmp3b4zidx5/model, flavor: spark). Fall back to return ['pyspark==4.3.0']. Set logging level to DEBUG to see the full traceback. 


Modèle enregistré avec signature corrigée !
Model URI : runs:/8cfe9d53c6c6412d8393df7e6b61a7fb/random_forest_tip_model_v2


In [0]:

import mlflow

# Utiliser le registre Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# URI de ton modèle enregistré
model_uri = (
    "runs:/8cfe9d53c6c6412d8393df7e6b61a7fb/"
    "random_forest_tip_model_v2"
)

# Nom du modèle dans Unity Catalog
model_name = "workspace.5lake_lab.random_forest_tip_model"

# Enregistrement
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Modèle enregistré dans Unity Catalog !")
print("Nom :", registered_model.name)
print("Version :", registered_model.version)


Successfully registered model 'workspace.5lake_lab.random_forest_tip_model'.


Uploading artifacts:   0%|          | 0/28 [00:00<?, ?it/s]

Modèle enregistré dans Unity Catalog !
Nom : workspace.5lake_lab.random_forest_tip_model
Version : 1


🔗 Created version '1' of model 'workspace.5lake_lab.random_forest_tip_model': https://dbc-7c4831e0-b044.cloud.databricks.com/explore/data/models/workspace/5lake_lab/random_forest_tip_model/version/1?o=7474659095651048


In [0]:

import mlflow
import mlflow.spark

mlflow.set_registry_uri("databricks-uc")

model_uri = "models:/workspace.5lake_lab.random_forest_tip_model/1"

# Chemin temporaire dans le Volume Unity Catalog
tmp_path = "/Volumes/workspace/5lake_lab/raw/mlflow_tmp"

# Charger le modèle enregistré
loaded_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir=tmp_path
)

print("Modèle Unity Catalog chargé avec succès !")


Modèle Unity Catalog chargé avec succès !


In [0]:

from pyspark.sql import functions as F

# Lire les données Silver
silver_df = spark.table(
    "workspace.5lake_lab.silver_trips_pipeline"
)

# Préparer les variables du modèle
scoring_df = (
    silver_df
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("tpep_pickup_datetime")
    )
    .dropna(subset=[
        "trip_distance",
        "trip_duration_min",
        "pickup_hour",
        "day_of_week"
    ])
)

# Appliquer le modèle enregistré
gold_predictions = loaded_model.transform(scoring_df)

# Ajouter la version du modèle
gold_predictions = gold_predictions.withColumn(
    "model_version",
    F.lit(1)
)

display(
    gold_predictions.select(
        "trip_distance",
        "tip_pct",
        "prediction",
        "model_version"
    ).limit(10)
)


trip_distance,tip_pct,prediction,model_version
2.89,24.14,15.658809270604758,1
0.6,23.87,15.859316420441255,1
0.92,12.14,15.116699010587727,1
2.14,11.61,14.701069473688545,1
3.02,2.58,15.178499032154015,1
1.17,8.27,15.33252224743907,1
1.9,27.31,14.7361014731314,1
1.8,2.68,14.830835119704423,1
20.22,15.74,15.123776349269354,1
1.2,22.34,15.370155351423337,1


In [0]:

from pyspark.sql import functions as F

# Charger les données du pipeline Silver
silver_df = spark.table(
    "workspace.5lake_lab.silver_trips_pipeline"
)

# Préparer les variables du modèle
scoring_df = (
    silver_df
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("tpep_pickup_datetime")
    )
    .dropna(subset=[
        "trip_distance",
        "trip_duration_min",
        "pickup_hour",
        "day_of_week"
    ])
)

# Prédire avec le modèle Unity Catalog
gold_predictions = loaded_model.transform(scoring_df)

# Ajouter la version utilisée
gold_predictions = gold_predictions.withColumn(
    "model_version",
    F.lit(1)
)

# Afficher les prédictions
display(
    gold_predictions.select(
        "trip_distance",
        "tip_pct",
        "prediction",
        "model_version"
    ).limit(10)
)


trip_distance,tip_pct,prediction,model_version
2.89,24.14,15.658809270604758,1
0.6,23.87,15.859316420441255,1
0.92,12.14,15.116699010587727,1
2.14,11.61,14.701069473688545,1
3.02,2.58,15.178499032154015,1
1.17,8.27,15.33252224743907,1
1.9,27.31,14.7361014731314,1
1.8,2.68,14.830835119704423,1
20.22,15.74,15.123776349269354,1
1.2,22.34,15.370155351423337,1


In [0]:

table_name = "workspace.5lake_lab.gold_predictions"

# Vérifier si la table existe déjà
if spark.catalog.tableExists(table_name):
    print("La table gold_predictions existe déjà.")
    print("Aucune modification effectuée.")
else:
    (
        gold_predictions.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    print("Table Gold créée avec succès !")


Table Gold créée avec succès !


In [0]:
%sql

SELECT
    trip_distance,
    tip_pct,
    prediction,
    model_version
FROM workspace.`5lake_lab`.gold_predictions
LIMIT 10;


trip_distance,tip_pct,prediction,model_version
2.89,24.14,15.658809270604758,1
0.6,23.87,15.859316420441255,1
0.92,12.14,15.116699010587727,1
2.14,11.61,14.701069473688545,1
3.02,2.58,15.178499032154015,1
1.17,8.27,15.33252224743907,1
1.9,27.31,14.7361014731314,1
1.8,2.68,14.830835119704423,1
20.22,15.74,15.123776349269354,1
1.2,22.34,15.370155351423337,1


In [0]:
%sql

SELECT COUNT(*) AS total_predictions
FROM workspace.`5lake_lab`.gold_predictions;


total_predictions
21847
